# QuantumTX AH — Deep EDA Notebook

Comprehensive exploratory analysis of the 2024 Alexandra Hospital dataset.
Covers responder profiling, dropout analysis, dosage-response, comorbidity
patterns, and feature correlations.

**Run top-to-bottom.** All artefacts (PNGs, CSVs) are saved to `reports/eda_artefacts/`.

---

## Section 0 — Setup & Data Load

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend — required for save_fig
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import Markdown, display

# Project root — works whether kernel CWD is project root or notebooks/
PROJECT_ROOT = Path(".").resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

ARTEFACTS = PROJECT_ROOT / "reports" / "eda_artefacts"
ARTEFACTS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
PALETTE = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2","#937860","#DA8BC3","#8C8C8C"]

def save_fig(name: str):
    """Save current matplotlib figure to artefacts dir and display inline."""
    plt.savefig(ARTEFACTS / name, dpi=150, bbox_inches="tight")
    print(f"Saved: {ARTEFACTS / name}")
    plt.show()
    plt.close("all")

print("Setup complete.")


In [ ]:
df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "featured.parquet")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Derived subsets used throughout
df_followup      = df[df["has_followup"] == "Y"].copy()
df_dropout_known = df[df["is_dropout"].notna()].copy()

# Column groups
HAS_FLAGS   = sorted([c for c in df.columns if c.startswith("has_") and c != "has_followup"])
GRP_FLAGS   = sorted([c for c in df.columns if c.startswith("grp_")])
RGN_FLAGS   = sorted([c for c in df.columns if c.startswith("rgn_")])
IMPROV_COLS = [c for c in ["vas_improvement","tug_improvement","sst_improvement",
                             "normal_gs_improvement","fast_gs_improvement","sppb_improvement"]
               if c in df.columns]
_ALL_IMPROV_LABELS = {
    "vas_improvement":       "VAS Pain",
    "tug_improvement":       "TUG",
    "sst_improvement":       "5xSST",
    "normal_gs_improvement": "Normal GS",
    "fast_gs_improvement":   "Fast GS",
    "sppb_improvement":      "SPPB",
}
IMPROV_LABELS = {k: v for k, v in _ALL_IMPROV_LABELS.items() if k in IMPROV_COLS}
# MCID thresholds. None = percentage-based (computed per-row in analysis cells).
MCID_THRESHOLDS = {
    "vas_improvement":       2.0,    # ≥2 point reduction in VAS (0-10 scale)
    "tug_improvement":       3.0,    # ≥3 s improvement in TUG
    "sst_improvement":       None,   # ≥10% of pre_5xsst_s (relative — see Section 4)
    "normal_gs_improvement": 0.05,   # ≥0.05 m/s improvement
    "fast_gs_improvement":   0.10,   # ≥0.10 m/s improvement
    "sppb_improvement":      1.0,    # ≥1 point improvement in SPPB
}
NUMERIC_FEATS = [c for c in ["age","baseline_sppb","pre_normal_gs_ms","pre_tug_s",
                               "pre_5xsst_s","pre_vas","pre_fast_gs_ms",
                               "n_flags","n_regions","n_groups"] if c in df.columns]

# Reproduce README overview stats
total_n      = len(df)
n_followup   = (df["has_followup"] == "Y").sum()
n_responders = int((df_followup["overall_responder"] == 1).sum())
print(f"\nOverview:")
print(f"  Total patients : {total_n:,}")
print(f"  With follow-up : {n_followup:,}  ({n_followup/total_n*100:.1f}%)")
print(f"  Responders     : {n_responders:,}  ({n_responders/n_followup*100:.1f}% of follow-up)")
print(f"  Dropouts       : {int(df['is_dropout'].sum()):,}")


---
## Section 1 — Cohort Profiles

How are patients distributed across cohorts? We profile each cohort by size,
age, gender, follow-up rate, and responder rate to establish baseline context
for all later analyses.

In [ ]:
# Build usage frequency short labels for column names
USAGE_MAP = {
    "Once (1x/week, one leg)":               "pct_1x",
    "Twice (2x/week, one leg per session)":  "pct_2x",
    "L+R 10 (20-min session, 10 min each leg)": "pct_lr",
}

rows = []
for cohort in sorted(df["cohort"].dropna().unique()):
    sub    = df[df["cohort"] == cohort]
    sub_fu = sub[sub["has_followup"] == "Y"]
    n_sub  = len(sub)
    row = {
        "cohort":            cohort,
        "n":                 n_sub,
        "pct_of_total":      round(n_sub / len(df) * 100, 1) if len(df) else float("nan"),
        "mean_age":          round(sub["age"].mean(), 1),
        "std_age":           round(sub["age"].std(), 1),
        "pct_female":        round((sub["gender"] == "F").sum() / n_sub * 100, 1) if n_sub else float("nan"),
        "n_followup":        len(sub_fu),
        "pct_followup":      round(len(sub_fu) / n_sub * 100, 1) if n_sub else float("nan"),
        "pct_responder":     round((sub_fu["overall_responder"] == 1).sum() / len(sub_fu) * 100, 1)
                             if len(sub_fu) >= 5 else float("nan"),  # suppress rate for very small sub-cohorts
    }
    # Usage frequency breakdown per cohort
    for freq_val, col_name in USAGE_MAP.items():
        row[col_name] = round((sub["usage_frequency"] == freq_val).sum() / n_sub * 100, 1) if n_sub else float("nan")
    rows.append(row)

cohort_df = pd.DataFrame(rows)
cohort_df.to_csv(ARTEFACTS / "cohort_profiles.csv", index=False)
display(cohort_df)


In [ ]:
cohort_colors = [PALETTE[i % len(PALETTE)] for i in range(len(cohort_df))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(cohort_df["cohort"], cohort_df["n"], color=cohort_colors)
axes[0].set_title("Patients per Cohort")
axes[0].set_ylabel("N")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(cohort_df["cohort"], cohort_df["pct_followup"], color=cohort_colors)
axes[1].axhline(n_followup / total_n * 100, color="red", linestyle="--", label="Overall avg")
axes[1].set_title("Follow-up Rate by Cohort")
axes[1].set_ylabel("% with Follow-up")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()

plt.tight_layout()
save_fig("cohort_overview.png")


**Note:** ~60% of patients are classified as Unclassified (no comorbidity tags assigned). Subgroup analyses within named cohorts (Frailty, Neurological, etc.) are based on smaller samples and should be interpreted with caution.

---
## Section 2 — Responder Rates by Subgroup

Which patient groups respond best? We compute the percentage of overall_responder
(among follow-up patients) for every meaningful subgroup: cohort, age band, gender,
usage frequency, and each comorbidity flag.

In [ ]:
def responder_rate(subdf):
    """% overall_responder among follow-up patients in subdf. Returns (rate, n_followup)."""
    fu = subdf[subdf["has_followup"] == "Y"]
    if len(fu) < 5:
        return float("nan"), len(fu)
    return round((fu["overall_responder"] == 1).sum() / len(fu) * 100, 1), len(fu)

rows = []
for val in sorted(df["cohort"].dropna().unique()):
    r, n = responder_rate(df[df["cohort"] == val])
    rows.append({"group": "cohort", "value": val, "n_followup": n, "responder_rate_pct": r})

for val in sorted(df["age_band"].dropna().unique()):
    r, n = responder_rate(df[df["age_band"] == val])
    rows.append({"group": "age_band", "value": str(val), "n_followup": n, "responder_rate_pct": r})

for val in [v for v in df["gender"].dropna().unique() if v != "__missing__"]:
    r, n = responder_rate(df[df["gender"] == val])
    rows.append({"group": "gender", "value": val, "n_followup": n, "responder_rate_pct": r})

for val in sorted(df["usage_frequency"].dropna().unique()):
    if val == "__missing__":
        continue
    r, n = responder_rate(df[df["usage_frequency"] == val])
    rows.append({"group": "usage_frequency", "value": str(val), "n_followup": n, "responder_rate_pct": r})

for flag in HAS_FLAGS:
    r, n = responder_rate(df[df[flag] == 1])
    rows.append({"group": "comorbidity_flag", "value": flag, "n_followup": n, "responder_rate_pct": r})

subgroup_df = pd.DataFrame(rows)
suppressed = subgroup_df[subgroup_df["responder_rate_pct"].isna()]
if len(suppressed):
    print(f"Suppressed {len(suppressed)} subgroups with n_followup < 5: {suppressed['value'].tolist()}")
subgroup_df = subgroup_df.dropna(subset=["responder_rate_pct"])
subgroup_df.to_csv(ARTEFACTS / "responder_rates_by_subgroup.csv", index=False)
display(subgroup_df.sort_values("responder_rate_pct", ascending=False).head(20))


In [ ]:
flag_df = (subgroup_df[subgroup_df["group"] == "comorbidity_flag"]
           .sort_values("responder_rate_pct", ascending=True))
overall_avg = (df_followup["overall_responder"] == 1).sum() / len(df_followup) * 100

fig, ax = plt.subplots(figsize=(8, max(5, len(flag_df) * 0.38)))
colors = ["#C44E52" if r < 50 else "#4C72B0" if r > 80 else "#DD8452"
          for r in flag_df["responder_rate_pct"]]
ax.barh(flag_df["value"].str.replace("has_", ""), flag_df["responder_rate_pct"], color=colors)
ax.axvline(overall_avg, color="black", linestyle="--", label=f"Overall avg ({overall_avg:.0f}%)")
ax.axvline(50, color="#C44E52", linestyle=":", alpha=0.6, label="50% threshold (low)")
ax.axvline(80, color="#4C72B0", linestyle=":", alpha=0.6, label="80% threshold (high)")
ax.set_xlabel("Responder Rate (%)")
ax.set_title("Responder Rate by Comorbidity Flag\n(red = <50%, orange = mid, blue = >80%)")
ax.legend(fontsize=8)
plt.tight_layout()
save_fig("responder_rates_by_flag.png")
